# Lesson 12: A real training run + controlling the writing

**Why:** a 40-minute training run doesn't belong in a notebook cell. If VS Code closes, everything is lost. Real projects train with a **script** that:

- prints progress as it goes,
- **saves checkpoints** (the model's numbers, written to a file) regularly, so the work is never lost,
- can **resume** after being stopped,
- keeps the **best** version (lowest validation loss) separately.

The script is [`src/tinygpt/train.py`](../src/tinygpt/train.py). It's the same 5-step loop from Lesson 2, plus saving. This run uses a **bigger model**: C = 128, context = 128 characters, about **830,000 parameters** (4× Lesson 10).

### How to run it (from the VS Code terminal)

| Command | What it does |
|---|---|
| `uv run python -m tinygpt.train` | train from scratch (about 40 min on this laptop) |
| `Ctrl + C` | stop at any time (progress up to the last report is saved) |
| `uv run python -m tinygpt.train --resume` | continue where it stopped |
| `uv run python -m tinygpt.sample --prompt "The motor"` | write text with the trained model |

It saves two files in `checkpoints/`: `tesla.pt` (latest, for resuming) and `tesla_best.pt` (best so far). This notebook uses the best one.

## 1. Load the trained model

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tinygpt.train import CHECKPOINT_DIR, load_model

model, tok, ckpt = load_model(CHECKPOINT_DIR / "tesla_best.pt")
last = ckpt["history"][-1]
print(f"parameters:      {model.num_params():,}")
print(f"settings:        {model.config}")
print(f"best checkpoint: step {ckpt['step']}, validation loss {ckpt['best_val']:.3f}")
print(f"training time:   {ckpt['minutes']:.0f} minutes")

## 2. The whole journey in one picture

In [ ]:
h = ckpt["history"]
plt.figure(figsize=(9, 4))
plt.plot([r["step"] for r in h], [r["train"] for r in h], "o-", label="train")
plt.plot([r["step"] for r in h], [r["val"] for r in h], "o-", label="validation")
for y, name in [(3.05, "frequencies only"), (2.41, "bigram"), (1.90, "4 heads + ffwd"), (1.51, "Lesson 10 GPT")]:
    plt.axhline(y, ls="--", lw=0.8, color="gray")
    plt.text(h[-1]["step"], y + 0.02, name, fontsize=8, color="gray", ha="right")
plt.ylim(1.0, 3.3); plt.xlabel("step"); plt.ylabel("loss")
plt.title("Training the GPT on Tesla"); plt.legend(); plt.show()

## 3. Temperature: how adventurous is the writing?

Before softmax, the scores are **divided by the temperature**:

$$p = \text{softmax}\left(\frac{\text{scores}}{\text{temperature}}\right)$$

- **Temperature < 1** makes differences **bigger**: the favorite gets even more likely. Writing is safer and more correct, but can get repetitive.
- **Temperature = 1**: the model's own probabilities, unchanged.
- **Temperature > 1** makes differences **smaller**: unlikely characters get more chances. Writing is more surprising, with more mistakes.

First, what does the model think comes next at two moments: in the middle of a word it's sure about (`"curren"`), and at the start of a new word (after a space)?

In [ ]:
for prompt in ["The alternating curren", "The alternating current "]:
    with torch.no_grad():
        logits, _ = model(torch.tensor([tok.encode(prompt)]))
    print(f"after {prompt!r}:")
    for temp in [0.5, 1.0, 1.5]:
        probs = F.softmax(logits[0, -1] / temp, dim=-1)
        top = probs.topk(5)
        shown = ",  ".join(f"{tok.decode([i.item()])!r} {p.item():.0%}" for p, i in zip(top.values, top.indices))
        print(f"  temperature {temp}:  {shown}")

**Predict:** we'll write 300 characters at temperatures **0.5**, **1.0** and **1.5**, all starting from the same prompt. Which one will contain the **most real English words**? Which one the **most invented words**?

In [ ]:
start = torch.tensor([tok.encode("The alternating current ")])
for temp in [0.5, 1.0, 1.5]:
    torch.manual_seed(42)
    out = model.generate(start, 300, temperature=temp)
    print(f"===== temperature {temp} =====")
    print(tok.decode(out[0].tolist()))
    print()

## 4. Top-k: cut off the long tail

Even when the model is fairly sure, the dozens of unlikely characters together can still add up to a few percent. Every so often one gets picked, and the text derails. **Top-k** keeps only the **k most likely** characters at each step and sets the rest to −∞ (the Lesson 7 trick).

In [ ]:
for k in [None, 5]:
    torch.manual_seed(42)
    out = model.generate(start, 300, temperature=1.0, top_k=k)
    print(f"===== temperature 1.0, top_k={k} =====")
    print(tok.decode(out[0].tolist()))
    print()

## 5. Try it yourself

Change the prompt, the temperature or top_k and run again. The prompt can only use characters from Tesla's book (no ć).

In [ ]:
prompt = "My experiments with high frequency currents"
temperature = 0.8
top_k = 10

torch.manual_seed(7)
out = model.generate(torch.tensor([tok.encode(prompt)]), 400, temperature=temperature, top_k=top_k)
print(tok.decode(out[0].tolist()))

## 6. Final check

In [ ]:
assert ckpt["best_val"] < 1.51, "the longer run should beat Lesson 10"
print(f"Lesson 12 complete. Best validation loss {ckpt['best_val']:.3f}.")

## Wrap-up question

If you used a model like this to write text for a **client's website**, which temperature would you choose, and why? What about for brainstorming **new product names**?